# Langchain test

In [1]:
import tools
from langchain.agents import create_agent
import os
from datetime import date, timedelta
from dotenv import load_dotenv
#pip install langchain-google-genai

In [2]:
agent = create_agent(
    model="google_genai:gemma-4-31b-it",
    tools=[
        tools.get_company,
        tools.spot_significant_dates,
        tools.get_article_context,
    ],
    system_prompt=f"""

Today's date is {date.today()}
The stock symbol that is used for this conversation is: {"NVDA"}
If the user asks about any other company just warn them.

You are a financial research assistant.

IMPORTANT: Conversation history is your primary source of information.

Tools:
1. get_company: Get detailed information about the company
2. spot_significant_dates: Get dates where significant event occurs. always look for the column "percent_change" and "percent_increase".
3. get_article_context: fetch articles given a specific time range. Check on the title, full_text, and linked_info columns and try to correlate with the articles.

Before using any tool:
1. Check previous assistant messages and tool results.
2. Determine whether the required information already exists.
3. Only call tools if the information is missing.

Do NOT call tools to retrieve information that has already been provided
earlier in the conversation.

Examples:

User:
"What happened to NVDA this June?"

Correct workflow:
- Use tools to find significant events and related news.

User:
"Why did they enter biomedical research?"
"Can you explain the June 23 event?"
"What does this mean?"

Do NOT call tools again.
Use the previous NVDA analysis and retrieved articles.

Only call tools when:
- The user asks EXPLICITLY about the company's information.
- The user asks about a new time period.
- The user requests updated information.
- Previous context does not contain enough evidence."""
)

In [3]:
def chat():
    messages = []
    print("Stock chatbot ready. Type 'quit' or 'exit' to stop.\n")
    
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit"):
            print("Goodbye!")
            break
        if not user_input:
            continue
        
        print(f"You: {user_input}\n")
        messages.append({"role": "user", "content": user_input})

        # stream instead of invoke — prints each step as it happens
        for chunk in agent.stream({"messages": messages}):
            for node, update in chunk.items():
                if node == "agent":
                    content = update["messages"][-1].content
                    if isinstance(content, list):
                        for block in content:
                            if block.get("type") == "thinking":
                                print(f"[Thinking]: {block['thinking'][:200]}...\n")
                            elif block.get("type") == "text" and block["text"]:
                                print(f"[Agent]: {block['text']}\n")
                    elif content:
                        print(f"[Agent]: {content}\n")

                elif node == "tools":
                    for msg in update["messages"]:
                        print(f"[Tool call]: {msg.name}()")
                        print(f"[Tool result]: {str(msg.content)[:300]}...\n")

        # update messages with final state
        result = agent.invoke({"messages": messages})
        messages = result["messages"]
        
        reply = result["messages"][-1].content
        if isinstance(reply, list):
            reply = " ".join(block["text"] for block in reply if block.get("type") == "text")
        
        print(f"Assistant: {reply}\n")

chat()

Stock chatbot ready. Type 'quit' or 'exit' to stop.

You: how was nvidia doing this june?

[Tool call]: get_article_context()
[Tool result]:                                                  url symbol  \
0  https://finance.yahoo.com/markets/stocks/artic...   NVDA   
1  https://finance.yahoo.com/markets/stocks/artic...   NVDA   
2  https://finance.yahoo.com/markets/stocks/artic...   NVDA   
3  https://finance.yahoo.com/markets/stocks/arti...

Assistant: In June 2026, NVIDIA (NVDA) experienced a period of volatility followed by a strong recovery:

*   **Early June Dip:** Around June 5th, NVDA and other major chip stocks (including Intel, AMD, and Micron) saw a decline.
*   **Mid-Month Rally:** The stock rebounded significantly by mid-month. On June 15th, NVDA reclaimed a **$5 trillion market cap**, with shares gaining roughly 6% over a short period. This surge was partly attributed to broader market optimism following a geopolitical breakthrough regarding a peace framework with Iran.
*  